In [1]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd

In [2]:
SEED = 42
np.random.seed(SEED)

declare global seed 

In [4]:
os.getcwd()

'D:\\local\\ds-aiml-set-c-10828\\notebook'

In [5]:
os.chdir('D:\\local\\ds-aiml-set-c-10828')

In [6]:
os.getcwd()

'D:\\local\\ds-aiml-set-c-10828'

change directory to make new folders for output and models

In [7]:
Path("outputs/figures").mkdir(parents=True, exist_ok=True)
Path("models").mkdir(parents=True, exist_ok=True)

#### Import csv data

In [8]:
print("--- Running Data Preprocessing & Splitting ---")
df_raw = pd.read_csv("data/raw/set_b.csv")
raw_shape = df_raw.shape
exact_duplicates_count = int(df_raw.duplicated().sum())

--- Running Data Preprocessing & Splitting ---


In [17]:
df_raw.head()

,record_id,distance,load,traffic,staff,group,late
0,1,68.12,42.71,39.14,45.98,G2,1
1,2,38.47,65.08,58.80,56.38,G1,1
2,3,64.10,43.99,36.86,53.47,G2,0
3,4,53.09,41.44,55.36,64.72,G1,1
4,5,44.67,51.95,53.44,54.86,G1,1


In [11]:
raw_shape

(305, 7)

In [12]:
exact_duplicates_count

5

#### Remove duplicates and get new clean df

In [9]:
# Drop exact duplicates
df_clean = df_raw.drop_duplicates().copy()
clean_shape = df_clean.shape
missing_counts = df_clean.isnull().sum().to_dict()
target_counts = df_clean["late"].value_counts().to_dict()

In [13]:
clean_shape

(300, 7)

In [14]:
missing_counts

{'record_id': 0,
 'distance': 15,
 'load': 15,
 'traffic': 0,
 'staff': 0,
 'group': 0,
 'late': 0}

In [15]:
target_counts

{0: 155, 1: 145}

#### Split data into training, testing and validation

In [16]:
from sklearn.model_selection import train_test_split

In [18]:
train_val_df, test_df = train_test_split(
    df_clean,
    test_size=0.20,
    random_state=SEED,
    stratify=df_clean["late"]
)

We use stratify to maintain class distribution

In [19]:
fit_df, val_df = train_test_split(
    train_val_df,
    test_size=0.20,
    random_state=SEED,
    stratify=train_val_df["late"]
)

#### Save Split dfs in output folder

In [20]:
# Save data ids
fit_ids = set(fit_df["record_id"])
val_ids = set(val_df["record_id"])
test_ids = set(test_df["record_id"])

In [21]:
is_disjoint = (
    len(fit_ids.intersection(val_ids)) == 0 and
    len(fit_ids.intersection(test_ids)) == 0 and
    len(val_ids.intersection(test_ids)) == 0
)

In [22]:
is_disjoint

True

In [23]:
splits_df = pd.DataFrame({
    "record_id": df_clean["record_id"],
    "partition": df_clean["record_id"].map(
        lambda rid: "fit" if rid in fit_ids else ("val" if rid in val_ids else "test")
    )
})
splits_df.to_csv("outputs/splits.csv", index=False)